# الفصل 16: نماذج الفضاء الحالة: SSM وMamba

> كل ما تعلمناه حتى الآن (RoPE، GQA، Flash Attention، MoE) هو تحسينات ذكية حول آلية Attention الأساسية، لكنها لا تزال تحمل جوهرها التربيعي (أو على الأقل تحتاج حلولًا معقدة لتجاوزه).

- **الجزء:** 4 — المعماريات الحديثة
- **الفصل:** 16
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 15: خليط الخبراء: Mixture of Experts
- **الوسوم:** SSM، Mamba، Selectivity، Jamba

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch16-state-space-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch16-state-space-models.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch16-state-space-models.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> كل ما تعلمناه حتى الآن (RoPE، GQA، Flash Attention، MoE) هو تحسينات ذكية حول آلية Attention الأساسية، لكنها لا تزال تحمل جوهرها التربيعي (أو على الأقل تحتاج حلولًا معقدة لتجاوزه).

> **بيانات الفصل**
>
> - **الجزء:** 4 — المعماريات الحديثة
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 15: خليط الخبراء: Mixture of Experts


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم SSM
- فهم Mamba
- فهم Selectivity


## 1. لماذا يهمّك هذا الموضوع؟

كل ما تعلمناه حتى الآن (RoPE، GQA، Flash Attention، MoE) هو **تحسينات ذكية** حول آلية Attention الأساسية، لكنها لا تزال تحمل جوهرها التربيعي (أو على الأقل تحتاج حلولًا معقدة لتجاوزه). هذا الفصل يُقدِّم شيئًا مختلفًا جذريًا: عائلة من المعماريات تُسمى **نماذج الفضاء الحالي (State Space Models - SSM)**، وعلى رأسها **Mamba**، التي **تتخلى تمامًا عن آلية Attention** وتستبدلها بآلية ذات **تعقيد خطي $O(n)$** بالنسبة لطول التسلسل — مستوحاة من نظرية التحكم الكلاسيكية (control theory) ومعاد تصميمها لتكون "انتقائية" (selective) وقابلة للتدريب على نطاق واسع بكفاءة تُضاهي Transformer.

---


## 2. الخريطة الذهنية


```text
الفكرة الجذرية: استبدال Attention بـ "حالة مخفية مستمرة التحديث"
(يُذكّرنا هذا بـ RNN، لكن بصياغة رياضية مختلفة تمامًا تسمح بالتوازي الكامل)

نماذج الفضاء الحالي الكلاسيكية (من نظرية التحكم):
│
├── معادلة الحالة:    h'(t) = A h(t) + B x(t)
├── معادلة الإخراج:   y(t) = C h(t) + D x(t)
│
└── A, B, C, D مصفوفات ثابتة (لا تتغير حسب المُدخَل!)
    → هذا قيد خطير: لا يمكن للنموذج "اختيار" ما يتذكره ديناميكيًا

ابتكار Mamba: الانتقائية (Selectivity)
│
├── اجعل B, C, وخطوة التقطيع Δ (delta) دوال من المُدخل x(t) نفسه
│   (وليست ثوابت مُتعلَّمة فقط)
│
├── هذا يسمح للنموذج بـ "اختيار" ديناميكيًا:
│   - ما المعلومات التي يجب تذكّرها من الحالة المخفية
│   - ما المعلومات التي يجب نسيانها/تجاهلها
│   (شبيه مفاهيميًا ببوابات LSTM، لكن بكفاءة حسابية أعلى بكثير)
│
└── خوارزمية Hardware-Aware Scan:
    تنفيذ ذكي (مشابه لفلسفة Flash Attention) يستغل بنية GPU
    لحساب التكرار بكفاءة عالية رغم الطبيعة التسلسلية ظاهريًا

النتيجة:
│
├── تعقيد زمني وذاكري خطي O(n) بدلًا من O(n²)
├── أداء يُضاهي Transformer في مهام اللغة على المقاييس المُختبَرة
└── استدلال أسرع بكثير للسياقات الطويلة جدًا (لا حاجة لـ KV Cache ضخم)

Jamba: نموذج هجين (Transformer + Mamba في نفس المعمارية)
يجمع بين كفاءة Mamba وقوة Attention الانتقائية في طبقات متناوبة
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل أنك تقرأ كتابًا طويلًا جدًا، وتحتفظ في ذهنك بـ"ملخص متجدد" (حالة مخفية) لما قرأته حتى الآن، بدلاً من إعادة قراءة كل صفحة سابقة في كل مرة تريد فهم الجملة الحالية (كما تفعل Attention حرفيًا). المشكلة في النماذج الكلاسيكية من هذا النوع (SSM التقليدية) أن "طريقة التلخيص" ثابتة دائمًا، بغض النظر عن محتوى ما تقرأه — وكأنك تُلخِّص رواية بوليسية وكتابًا علميًا بنفس الطريقة والمعايير تمامًا.

ابتكار Mamba هو جعل "طريقة التلخيص" نفسها **تتكيف ديناميكيًا** حسب محتوى كل جملة تقرأها: عندما تقرأ معلومة حاسمة (مثل اسم الجاني في رواية بوليسية)، "تنتبه" بقوة وتُحدِّث ملخصك الذهني بشكل كبير؛ وعندما تقرأ جملة انتقالية غير مهمة، تحافظ على ملخصك كما هو تقريبًا دون تغيير يُذكر. هذه "الانتقائية" الديناميكية هي بالضبط ما يُميّز Mamba عن أسلافه من نماذج SSM.

### 🟡 طبقة الآلية: من SSM الكلاسيكي إلى Mamba

**نموذج الفضاء الحالي المستمر (الصيغة الأصلية من نظرية التحكم)**:

$$h'(t) = Ah(t) + Bx(t)$$
$$y(t) = Ch(t) + Dx(t)$$

حيث $h(t)$ الحالة المخفية المستمرة، و$A, B, C, D$ مصفوفات ثابتة تُحدِّد كيف تتطور الحالة وكيف تُترجَم لإخراج.

**التقطيع (Discretization)**: بما أن بياناتنا (اللغة) متقطعة (discrete) وليست مستمرة، نحتاج لتحويل هذه المعادلة التفاضلية المستمرة لصيغة تكرارية متقطعة، باستخدام خطوة زمنية $\Delta$:

$$\bar{A} = \exp(\Delta A), \qquad \bar{B} = (\Delta A)^{-1}(\exp(\Delta A) - I) \cdot \Delta B$$

$$h_t = \bar{A}h_{t-1} + \bar{B}x_t$$
$$y_t = Ch_t$$

هذه الصيغة التكرارية تُذكّرنا بـ RNN، لكن المفتاح هو أنه يمكن أيضًا إعادة كتابتها كـ**عملية التفاف (convolution)** يمكن حسابها بالتوازي الكامل أثناء التدريب (عبر خوارزمية تحويل فورييه السريع أو تقنيات مشابهة) — هذا يحل مشكلة التسلسلية البطيئة لـ RNN الكلاسيكي أثناء التدريب، رغم بقاء الاستدلال تسلسليًا بطبيعته (لكن بكفاءة خطية ثابتة لكل خطوة).

**مشكلة SSM الكلاسيكي (S4 وما قبله)**: المصفوفات $A, B, C$ **ثابتة** (time-invariant) — لا تتغير حسب المدخل $x_t$ الفعلي. هذا يعني أن النموذج يُعامل كل المدخلات بنفس "القواعد" الديناميكية، بغض النظر عن محتواها — قيد كبير جدًا مقارنة بـ Attention الذي يحسب أوزانًا ديناميكية مختلفة تمامًا لكل زوج من الكلمات حسب محتواها الفعلي.

**ابتكار Mamba — الانتقائية (Selective SSM / S6)**: اجعل $B$, $C$, و$\Delta$ **دوالًا من المُدخَل** $x_t$ نفسه (وليست ثوابت تمامًا):

$$B_t = \text{Linear}_B(x_t), \quad C_t = \text{Linear}_C(x_t), \quad \Delta_t = \text{softplus}(\text{Linear}_\Delta(x_t))$$

الآن تُصبح المعادلة:

$$h_t = \bar{A}_t h_{t-1} + \bar{B}_t x_t$$

حيث $\bar{A}_t, \bar{B}_t$ تعتمدان على $\Delta_t$ المُحسَّبة ديناميكيًا لكل رمز. هذا يعني أن قيمة $\Delta_t$ تتحكم فعليًا في "مدى أهمية" الرمز الحالي: $\Delta_t$ كبيرة → تحديث كبير للحالة (الرمز الحالي "مهم" ويُدمَج بقوة)؛ $\Delta_t$ صغيرة → الحالة تبقى شبه ثابتة (الرمز الحالي "أقل أهمية"، يشبه "تخطيه" دون تأثير يُذكر).

**التحدي الهندسي**: بجعل $A, B$ تعتمدان على المُدخَل، نفقد إمكانية استخدام خوارزمية الالتفاف السريعة (لأن المصفوفات لم تعد ثابتة عبر الزمن). حل Mamba لهذا هو خوارزمية **Hardware-Aware Parallel Scan** — تقنية ذكية مستوحاة من فلسفة Flash Attention، تستغل بنية ذاكرة GPU (SRAM السريعة مقابل HBM البطيئة) لحساب هذا التكرار بكفاءة عالية جدًا رغم طبيعته التسلسلية الظاهرية.

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================
# 1. SimplifiedMambaBlock — تطبيق مبسّط تعليمي
#    (التطبيق الفعلي الرسمي يستخدم CUDA kernels مُحسَّنة جدًا
#     لخوارزمية Parallel Scan؛ هذا الكود لفهم الآلية حسابيًا)
# ============================================================
class SimplifiedMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4,
                 expand_factor: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = expand_factor * d_model   # توسيع البعد الداخلي

        # إسقاط المدخل مع توسيع البعد (نمط "Gated" شائع)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # التفاف 1D محلي قصير المدى (يلتقط سياقًا محليًا قبل SSM)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            groups=self.d_inner, padding=d_conv - 1, bias=True
        )

        # إسقاطات لحساب B, C, Δ بشكل ديناميكي من المدخل (الانتقائية!)
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        # مصفوفة A (قابلة للتعلم لكن لا تعتمد مباشرة على x — إعداد S4/S6 القياسي)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float().repeat(self.d_inner, 1)))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        """ x: (batch, seq_len, d_model) """
        batch_size, seq_len, _ = x.shape

        # 1. الإسقاط والتوسيع + بوابة (gate)
        x_and_gate = self.in_proj(x)   # (batch, seq_len, 2*d_inner)
        x_inner, gate = x_and_gate.chunk(2, dim=-1)

        # 2. التفاف محلي قصير المدى
        x_conv = x_inner.transpose(1, 2)              # (batch, d_inner, seq_len)
        x_conv = self.conv1d(x_conv)[:, :, :seq_len]   # اقتصاص padding الزائد
        x_conv = x_conv.transpose(1, 2)                # (batch, seq_len, d_inner)
        x_conv = F.silu(x_conv)   # تفعيل SiLU/Swish

        # 3. حساب B, C, Δ بشكل ديناميكي (الانتقائية الجوهرية لـ Mamba)
        x_dbl = self.x_proj(x_conv)   # (batch, seq_len, 2*d_state + 1)
        delta, B, C = torch.split(x_dbl, [1, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))   # (batch, seq_len, d_inner)

        A = -torch.exp(self.A_log)   # (d_inner, d_state)، سالبة لضمان استقرار النظام

        # 4. المسح التكراري (Selective Scan) — هنا التطبيق الفعلي يستخدم
        #    CUDA kernel مُحسَّن جدًا؛ هنا حلقة تسلسلية توضيحية فقط
        h = torch.zeros(batch_size, self.d_inner, self.d_state, device=x.device)
        outputs = []

        for t in range(seq_len):
            delta_t = delta[:, t, :].unsqueeze(-1)              # (batch, d_inner, 1)
            A_bar = torch.exp(delta_t * A.unsqueeze(0))          # (batch, d_inner, d_state)
            B_t = B[:, t, :].unsqueeze(1)                        # (batch, 1, d_state)
            x_t = x_conv[:, t, :].unsqueeze(-1)                  # (batch, d_inner, 1)

            h = A_bar * h + delta_t * B_t * x_t                  # تحديث الحالة

            C_t = C[:, t, :].unsqueeze(1)                        # (batch, 1, d_state)
            y_t = (h * C_t).sum(dim=-1)                          # (batch, d_inner)
            outputs.append(y_t)

        y = torch.stack(outputs, dim=1)   # (batch, seq_len, d_inner)
        y = y + x_conv * self.D   # اتصال متبقٍّ مباشر (skip connection) بوزن D

        # 5. البوابة النهائية (Gating) ثم الإسقاط للخارج
        y = y * F.silu(gate)
        output = self.out_proj(y)

        return output

# ============================================================
# 2. طبقة Mamba كاملة (مع LayerNorm والاتصال المتبقي)
# ============================================================
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand_factor=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = SimplifiedMambaBlock(d_model, d_state, d_conv, expand_factor)

    def forward(self, x):
        return x + self.mamba(self.norm(x))

# ============================================================
# 3. نموذج Mamba كامل مُكدَّس (بديل كامل لـ GPT من الفصل 12)
# ============================================================
class MambaLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_layers=6, d_state=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight   # Weight Tying

    def forward(self, input_ids, targets=None):
        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x)   # ملاحظة: لا حاجة لأي قناع (mask)! السببية متأصلة في التصميم

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                     targets.view(-1), ignore_index=-100)
        return logits, loss

# ============================================================
# 4. مفهوم Jamba — معمارية هجينة (Transformer + Mamba)
# ============================================================
class JambaStyleBlock(nn.Module):
    """
    كتلة هجينة تتناوب بين Mamba (للكفاءة والسياق الطويل)
    وMulti-Head Attention (للقدرة على الربط الدقيق بين عناصر بعيدة محددة).
    """
    def __init__(self, d_model, num_heads, d_ff, use_attention: bool,
                 d_state=16, num_experts=8, top_k=2):
        super().__init__()
        self.use_attention = use_attention

        if use_attention:
            self.norm1 = nn.LayerNorm(d_model)
            self.mixer = MultiHeadAttention(d_model, num_heads)
        else:
            self.norm1 = nn.LayerNorm(d_model)
            self.mixer = SimplifiedMambaBlock(d_model, d_state)

        self.norm2 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, d_ff, num_experts, top_k)  # من الفصل 15

    def forward(self, x, mask=None):
        normed = self.norm1(x)
        if self.use_attention:
            mix_out, _ = self.mixer(normed, normed, normed, mask)
        else:
            mix_out = self.mixer(normed)
        x = x + mix_out

        normed = self.norm2(x)
        moe_out, aux_loss = self.moe(normed)
        x = x + moe_out

        return x, aux_loss

def build_jamba_style_model(num_layers=8, attention_ratio=0.25):
    """
    بناء تسلسل من الطبقات حيث نسبة صغيرة فقط (مثلًا 1 من كل 4، كما في Jamba)
    تستخدم Attention الكامل، والباقي يستخدم Mamba الأكفأ.
    """
    attention_positions = set(
        int(i / attention_ratio) for i in range(int(num_layers * attention_ratio))
    )
    layer_types = ["attention" if i in attention_positions else "mamba"
                   for i in range(num_layers)]
    return layer_types


---


## 4. مثال محلول خطوة بخطوة

لنحاكِ خطوة واحدة من Selective Scan بأرقام مبسّطة جدًا (بُعد حالة $d_{state}=1$ للتبسيط الأقصى).

**القيم المُعطاة**:
- الحالة السابقة: $h_{t-1} = 0.5$
- قيمة $A = -1.0$ (سالبة دائمًا لضمان الاستقرار)
- المُدخَل الحالي: $x_t = 2.0$
- قيم ديناميكية محسوبة من المُدخَل: $\Delta_t = 0.3$, $B_t = 0.8$, $C_t = 1.2$

**الخطوة 1 — تقطيع $A$**:
$$\bar{A}_t = \exp(\Delta_t \cdot A) = \exp(0.3 \times -1.0) = \exp(-0.3) \approx 0.7408$$

**الخطوة 2 — تحديث الحالة**:
$$h_t = \bar{A}_t \cdot h_{t-1} + \Delta_t \cdot B_t \cdot x_t$$
$$h_t = 0.7408 \times 0.5 + 0.3 \times 0.8 \times 2.0 = 0.3704 + 0.48 = 0.8504$$

**الخطوة 3 — حساب الإخراج**:
$$y_t = C_t \cdot h_t = 1.2 \times 0.8504 \approx 1.0205$$

**لنُقارن الآن بسيناريو مختلف حيث $\Delta_t$ صغيرة جدًا (0.01) — أي "الرمز الحالي غير مهم"**:

$$\bar{A}_t = \exp(0.01 \times -1.0) = \exp(-0.01) \approx 0.99$$
$$h_t = 0.99 \times 0.5 + 0.01 \times 0.8 \times 2.0 = 0.495 + 0.016 = 0.511$$

**ملاحظة جوهرية**: في هذه الحالة الثانية، الحالة الجديدة $h_t=0.511$ قريبة جدًا من الحالة القديمة $h_{t-1}=0.5$ — أي أن المُدخَل $x_t$ لم يُؤثر عمليًا تقريبًا على الحالة، رغم أن قيمته (2.0) لم تتغير! هذا بالضبط ما نعنيه بـ"الانتقائية": قيمة $\Delta_t$ (المحسوبة ديناميكيًا من محتوى $x_t$ نفسه عبر طبقة خطية متعلَّمة) هي التي تُقرر "كم يجب أن نهتم" بهذا المُدخَل المحدد، بغض النظر عن قيمته العددية الخام — تمامًا كبوابة النسيان/الإدخال في LSTM، لكن بصياغة رياضية مختلفة تسمح بتوازٍ حسابي أكفأ.

---


## 5. نصائح وحيل

> 💡 **استخدم التطبيقات الرسمية المُحسَّنة دائمًا في الإنتاج**: الكود أعلاه (بحلقة Python تسلسلية) لأغراض تعليمية بحتة؛ مكتبة `mamba-ssm` الرسمية تستخدم CUDA kernels مُحسَّنة جدًا (Hardware-Aware Scan) أسرع بعشرات المرات.

> 💡 **Mamba يتفوق خصوصًا في السياقات الطويلة جدًا**: الفرق في السرعة والذاكرة بين Mamba وTransformer يُصبح واضحًا بشدة فقط عند أطوال سياق كبيرة جدًا (عشرات أو مئات الآلاف من الرموز)؛ لسياقات قصيرة إلى متوسطة، التفوق أقل وضوحًا، وقد يتفوق Transformer المحسَّن (Flash Attention + GQA) عمليًا.

> 💡 **المعماريات الهجينة (مثل Jamba) غالبًا الخيار الأذكى عمليًا**: الجمع بين كفاءة Mamba (لمعظم الطبقات) وقوة Attention الدقيقة (لنسبة صغيرة من الطبقات، مثل 1 من كل 8) يحقق توازنًا ممتازًا بين الكفاءة الحسابية والقدرة على الربط الدقيق بين عناصر بعيدة محددة التي قد يُصعِّبها ضغط المعلومات في حالة مخفية واحدة.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **الاعتقاد بأن Mamba هو "RNN آخر" بنفس قيود RNN القديمة**: رغم الصياغة التكرارية الظاهرية، فإن Mamba (وSSM عمومًا) يمكن تدريبها بالتوازي الكامل عبر خوارزميات الـ scan أو الالتفاف الخاصة، على عكس RNN التقليدي الذي يتطلب تسلسلية صارمة أثناء التدريب أيضًا — هذا فرق هندسي جوهري غالبًا ما يُساء فهمه.

> ⚠️ **تجاهل أهمية $A$ السالبة لضمان الاستقرار**: إذا سُمح لـ $A$ (أو $\bar{A}$ الناتجة) بأن تكون موجبة أو كبيرة جدًا، فإن الحالة المخفية قد "تنفجر" (exponential blow-up) عبر خطوات زمنية طويلة — تهيئة $A_{log}$ واستخدام $-\exp(A_{log})$ لضمان السالبية الدائمة تصميم متعمد وضروري.

> ⚠️ **المقارنة غير العادلة بين Mamba وTransformer دون ضبط العتاد والتحسينات بالتساوي**: كثير من الادعاءات حول "تفوق Mamba الساحق" في الأوراق المبكرة قورنت أحيانًا بتطبيقات Transformer غير محسَّنة (بدون Flash Attention مثلًا)؛ المقارنات العادلة تتطلب أفضل تطبيق مُحسَّن من كل جانب.

---


## 7. جدول مقارنة

| الخاصية | Transformer (مع Attention) | Mamba (SSM الانتقائي) |
|---|---|---|
| **التعقيد الزمني (تدريب)** | $O(n^2)$ | $O(n)$ |
| **التعقيد الذاكري (استدلال)** | يحتاج KV Cache ينمو خطيًا | حالة مخفية بحجم ثابت! |
| **القدرة على الربط الدقيق البعيد المدى** | ممتازة (انتباه مباشر صريح) | جيدة لكن عبر ضغط في حالة واحدة |
| **قابلية التوازي أثناء التدريب** | كاملة وطبيعية | ممكنة عبر خوارزميات scan خاصة |
| **الأداء على سياقات قصيرة-متوسطة** | ممتاز، ناضج جدًا | جيد جدًا، قريب أو مُضاهٍ |
| **الأداء على سياقات طويلة جدًا جدًا** | مكلف جدًا رغم التحسينات | أكفأ بكثير نظريًا وعمليًا |
| **النضج والدعم في الصناعة (2024)** | ناضج جدًا، دعم شامل | ناشئ، يتطور بسرعة |

---


## 8. ملخص أفضل الممارسات

- ✅ فكّر في Mamba أو المعماريات الهجينة عندما يكون طول السياق المستهدف كبيرًا جدًا (أكثر من 32K-100K رمز) وتُشكّل كفاءة الاستدلال أولوية حرجة.
- ✅ استخدم التطبيقات الرسمية المُحسَّنة (`mamba-ssm`) بدلاً من إعادة التنفيذ اليدوي في أي استخدام إنتاجي.
- ✅ فكّر في المعماريات الهجينة (Jamba-style) كخيار أكثر توازنًا من الاعتماد الكلي على Mamba أو Transformer فقط.
- ✅ لا تفترض تفوق Mamba تلقائيًا لكل تطبيق — قيّم الحاجة الفعلية لطول السياق ومتطلبات الدقة قبل الاختيار.

---


## 9. أسئلة مقابلات

**س1: ما المقصود بـ"الانتقائية" (Selectivity) في Mamba، ولماذا هي التحسين الجوهري مقارنة بنماذج SSM الكلاسيكية السابقة مثل S4؟**

**الإجابة النموذجية**: في نماذج SSM الكلاسيكية (مثل S4)، تكون مصفوفات التحويل $A, B, C$ ثابتة تمامًا (time-invariant) — لا تتغير أبدًا بناءً على محتوى المدخل الفعلي، بل تُتعلَّم مرة واحدة أثناء التدريب وتبقى كما هي لكل رمز في كل تسلسل. هذا يعني أن النموذج يُطبِّق "نفس قاعدة التحديث" ديناميكيًا على كل المدخلات بغض النظر عن أهميتها النسبية — قيد جوهري يُضعف القدرة على التمييز بين المعلومات الحاسمة والمعلومات الثانوية. الابتكار الجوهري في Mamba هو جعل بعض هذه المصفوفات (تحديدًا $B$, $C$, وخطوة التقطيع $\Delta$) **دوالًا من المُدخَل الحالي نفسه** عبر طبقات خطية متعلَّمة صغيرة. هذا يسمح للنموذج بأن "يُقرر" ديناميكيًا، لكل رمز على حدة: هل هذا الرمز مهم بما يكفي ليُحدِّث الحالة المخفية بشكل كبير (قيمة $\Delta$ كبيرة)، أم أنه أقل أهمية ويمكن "تمريره" دون تأثير يُذكر على الذاكرة المتراكمة (قيمة $\Delta$ صغيرة)؟ هذه القدرة على "اختيار ما يُتذكَّر" ديناميكيًا (مشابهة مفاهيميًا لبوابات LSTM لكن بصياغة رياضية تسمح بتوازٍ حسابي أكبر بكثير) هي السبب الرئيسي وراء تفوق Mamba الكبير على أسلافها من SSM في مهام اللغة الطبيعية.

**س2: لماذا يُعتبر التعقيد الخطي $O(n)$ لـ Mamba ميزة حاسمة خصوصًا للاستدلال (inference) على سياقات طويلة جدًا، مقارنة بـ Transformer حتى مع Flash Attention؟**

**الإجابة النموذجية**: حتى مع Flash Attention (الذي يُحسِّن كفاءة **تنفيذ** Attention دون تغيير نتيجته الرياضية)، يبقى التعقيد **الخوارزمي** الأساسي لـ Attention تربيعيًا $O(n^2)$، وأهم من ذلك أن الاستدلال التسلسلي (autoregressive generation) يتطلب الاحتفاظ بـ KV Cache كامل لكل الرموز السابقة، وهذا الكاش ينمو **خطيًا** مع طول السياق، مما يعني استهلاكًا ذاكريًا متزايدًا باستمرار مع كل رمز جديد يُولَّد، وتباطؤًا تدريجيًا في سرعة التوليد مع تراكم السياق (لأن كل رمز جديد يحتاج الانتباه لعدد متزايد من الرموز السابقة). في المقابل، فإن Mamba يحتفظ بـ **حالة مخفية ذات حجم ثابت تمامًا** بغض النظر عن طول السياق المُعالَج حتى تلك اللحظة — لا حاجة لأي "كاش" ينمو؛ كل المعلومات التاريخية مضغوطة في حالة واحدة بحجم معروف مسبقًا وثابت. هذا يعني أن سرعة توليد الرمز التالي في Mamba تبقى **ثابتة تقريبًا** بغض النظر عن طول السياق الذي سبقه، بينما تتباطأ تدريجيًا في Transformer — فرق جوهري وحاسم في أي تطبيق يتطلب توليدًا سريعًا ومستمرًا على سياقات طويلة جدًا (مثل المحادثات الممتدة أو معالجة مستندات ضخمة).

---


## 10. تمرين عملي

1. نفّذ `SimplifiedMambaBlock` وطبّقه على تسلسل عشوائي قصير (مثلًا 16 رمزًا)، وتحقق من شكل المخرج النهائي مطابق لشكل المدخل.
2. عدّل القيم اليدوية في المثال المحلول خطوة بخطوة لتجربة قيم $\Delta_t$ مختلفة (من 0.001 إلى 1.0)، وارسم كيف تتغير نسبة مساهمة $h_{t-1}$ مقابل $x_t$ في الحالة الجديدة $h_t$ مع تغيّر $\Delta_t$.
3. قارن (ذهنيًا أو عبر كود بسيط) استهلاك الذاكرة النظري بين Transformer (مع KV Cache) وMamba (حالة ثابتة) عند توليد 10,000 رمز متتالٍ لنموذج بنفس حجم d_model تقريبًا.

---


## 11. مراجع للتعمّق

- Gu, A., & Dao, T. (2023). *Mamba: Linear-Time Sequence Modeling with Selective State Spaces*.
- Gu, A., Goel, K., & Ré, C. (2021). *Efficiently Modeling Long Sequences with Structured State Spaces* (S4).
- Lieber, O., et al. (2024). *Jamba: A Hybrid Transformer-Mamba Language Model*. AI21 Labs.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المعماريات الحديثة / الفصل 16: نماذج الفضاء الحالة: SSM وMamba.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
